# Solution 08 — the offline proof

Both parts as plain `assert`s, with zero API keys. If an assert fails, the notebook fails loudly,
which is exactly how it works under `pytest` in CI.

In [ ]:
from learningdantic import setup

MODEL = setup()

In [ ]:
from pydantic_ai import Agent, capture_run_messages, models
from pydantic_ai.messages import ModelResponse, TextPart, ToolCallPart, ToolReturnPart
from pydantic_ai.models.function import FunctionModel
from pydantic_ai.models.test import TestModel


def scripted_model(messages, info):
    """Call add(2, 3) first; answer in text once the tool result is back."""
    tool_ran = any(
        isinstance(part, ToolReturnPart) for message in messages for part in message.parts
    )
    if tool_ran:
        return ModelResponse(parts=[TextPart(content="The sum is 5.")])
    return ModelResponse(parts=[ToolCallPart(tool_name="add", args={"a": 2, "b": 3})])


spy = {"add_calls": 0}
offline_agent = Agent(FunctionModel(scripted_model), instructions="Do arithmetic with your tools.")


@offline_agent.tool_plain
def add(a: int, b: int) -> int:
    """Add two integers."""
    spy["add_calls"] += 1  # the spy: counts real executions
    return a + b

In [ ]:
# Part 1 — behaviour: the tool really ran once, and the call came before its result.
spy["add_calls"] = 0  # reset, so re-running this cell works
with capture_run_messages() as messages:
    r = await offline_agent.run("what is 2+3?")

kinds = [type(part).__name__ for message in messages for part in message.parts]
print("1. spy:", spy, "| flow:", kinds)

assert spy["add_calls"] == 1, f"add ran {spy['add_calls']} times, expected 1"
assert kinds.index("ToolCallPart") < kinds.index("ToolReturnPart"), kinds
assert r.output == "The sum is 5.", r.output

In [ ]:
# Part 2 — scope, offline: under the kill switch, check which tools the model was offered.
test_model = TestModel()
with models.override_allow_model_requests(False):  # a real model request would raise here
    await offline_agent.run("what is 2+3?", model=test_model)

offered = [tool.name for tool in test_model.last_model_request_parameters.function_tools]
print("2. the model was offered:", offered)

assert offered == ["add"], offered

**Notes**

- The spy counter is the simplest fake there is. For tools that call a service, hand the agent fake
  deps with `override(deps=...)` (notebook 08's `override(deps=...)` cell) and assert the fake was used.
- `override_allow_model_requests(False)` around a whole test suite turns "no paid model calls"
  into a guarantee. Fake models (`TestModel`, `FunctionModel`) are exempt.